В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [1]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 46.4 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [2]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

In [3]:
print(list(api.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


**word2vec-google-news-300**

*   Google News
*   3 млн слов/фраз; векторы размерности 300
*   Подходят для задач семантического поиска, аналогий, классификации текстов на английском языке

**word2vec-ruscorpora-300**


*   НКРЯ
*   185 тыс. слов; векторы размерности 300
*   Подходит для анализа русскоязычных текстов: семантическая близость, морфологический анализ, дообучение под конкретные задачи.

**fasttext-wiki-news-subwords-300**


*   Wikipedia 2017 + UMBC webbase + statmt.org news
*   1 млн векторов; размерность 300
*   FastText-эмбеддинги с subword information. Позволяют получать векторы для редких слов и токенов через разбиение на n-граммы символов. Полезны для работы с опечатками, неологизмами.




**Базовые операции с векторами**

In [5]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [6]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [20]:
model_2 = api.load("word2vec-google-news-300")

print(f"Размер словаря: {len(model_2.key_to_index)}")
print(f"Размерность векторов: {model_2.vector_size}")

[==================================================] 100.0% 1662.8/1662.8MB downloaded
Размер словаря: 3000000
Размерность векторов: 300


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов


In [28]:
def find_similar(model, token, topn):
  for word, score in model.most_similar(token, topn):
     print(f"{word}: {score:.4f}")

find_similar(model=model_2, token='rowing', topn=10)

Rowing: 0.5715
rowers: 0.5324
Rowing_Club: 0.4894
sculling: 0.4821
flatwater: 0.4767
Concept2: 0.4598
Devin_Swett: 0.4491
Rowers: 0.4389
coxless: 0.4384
swimming: 0.4364


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [22]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [23]:
skipgram_model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,      # размерность векторов
    window=3,             # размер контекстного окна
    min_count=1,          # минимальная частота слова
    workers=2,            # количество ядер
    sg=1                  # 1 = Skip-Gram
)

print(f"Размер словаря: {len(skipgram_model.wv.key_to_index)}")

Размер словаря: 65


In [25]:
print(f"Слова в словаре: {list(skipgram_model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [26]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = skipgram_model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [36]:
similar = skipgram_model.wv.most_similar('духовка', topn=5)
for word, score in similar:
  print(f"  {word}: {score:.4f}")

  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142


In [37]:
similar = skipgram_model.wv.most_similar('овощи', topn=5)
for word, score in similar:
  print(f"  {word}: {score:.4f}")

  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [38]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [51]:
similar = ft_model.wv.most_similar('духовка', topn=5)
for word, score in similar:
  print(f"  {word}: {score:.4f}")

  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944


In [52]:
similar = ft_model.wv.most_similar('варить', topn=5)
for word, score in similar:
  print(f"  {word}: {score:.4f}")

  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622


In [53]:
similar = ft_model.wv.most_similar('овощи', topn=5)
for word, score in similar:
  print(f"  {word}: {score:.4f}")

  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [43]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = skipgram_model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('словощи')
compare_models('вотка')
compare_models('дудка')


Сравнение для слова: 'словощи'
  Word2Vec: слово не найдено
  FastText: ['овощи', 'фольга']

Сравнение для слова: 'вотка'
  Word2Vec: слово не найдено
  FastText: ['кофе', 'завтрак']

Сравнение для слова: 'дудка'
  Word2Vec: слово не найдено
  FastText: ['барбекю', 'говядина']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [44]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [45]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [46]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [47]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [48]:
compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [49]:
compare_documents(1, 0)
compare_documents(1, 2)
compare_documents(1, 3)
compare_documents(1, 4)

Схожесть doc_1 и doc_0: 0.2735
  doc_1: deep learning uses neural networks
  doc_0: machine learning is interesting
Схожесть doc_1 и doc_2: -0.0573
  doc_1: deep learning uses neural networks
  doc_2: python programming for data science
Схожесть doc_1 и doc_3: 0.2031
  doc_1: deep learning uses neural networks
  doc_3: artificial intelligence is amazing
Схожесть doc_1 и doc_4: -0.2546
  doc_1: deep learning uses neural networks
  doc_4: computer vision processes images




```
Схожесть doc_1 и doc_0: 0.2735
  doc_1: deep learning uses neural networks
  doc_0: machine learning is interesting
```




10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [50]:
ft_model_10 = FastText(
    sentences=cooking_sentences,
    vector_size=10,
    window=3,
    min_count=1,
    workers=2
)

ft_model_50 = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

ft_model_100 = FastText(
    sentences=cooking_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=2
)

In [ ]:
similar = ft_model.wv.most_similar('варить', topn=5)
for word, score in similar:
  print(f"  {word}: {score:.4f}")

In [63]:
def test_model_dimensions(models_dict, test_words, topn=5):
    """
    Параметры:
    -----------
    models_dict : dict
        Словарь {название_размерности: модель}
    test_words : list
        Список слов для тестирования
    topn : int
        Количество похожих слов для вывода
    """

    for word in test_words:
        print(f"--------\n\nслово: {word}")
        for dim_name, model in models_dict.items():
            print(f"\nРазмерность {dim_name}:")

            for sim_word, score in model.wv.most_similar(word, topn=topn):
              print(f"  {sim_word}: {score:.4f}")


In [64]:
models = {
    "10": ft_model_10,
    "50": ft_model_50,
    "100": ft_model_100
}

test_words = ["кастрюля", "сыр", "помидор"]
test_model_dimensions(models, test_words, topn=5)

--------

слово: кастрюля

Размерность 10:
  барбекю: 0.5832
  сковорода: 0.5718
  месить: 0.5516
  сливки: 0.5101
  уголь: 0.4468

Размерность 50:
  лимон: 0.3262
  запекать: 0.1596
  яйца: 0.1455
  хлеб: 0.1445
  ингредиенты: 0.1417

Размерность 100:
  яичница: 0.3324
  торт: 0.2023
  вино: 0.1752
  дрожжи: 0.1503
  соль: 0.1436
--------

слово: сыр

Размерность 10:
  тушить: 0.6884
  тесто: 0.6095
  сахар: 0.5359
  мариновать: 0.5043
  специи: 0.4392

Размерность 50:
  рыба: 0.4788
  специи: 0.2427
  месить: 0.2389
  вино: 0.2344
  чай: 0.2158

Размерность 100:
  горшок: 0.2098
  соус: 0.1789
  специи: 0.1619
  начинка: 0.1604
  взбивать: 0.1427
--------

слово: помидор

Размерность 10:
  помидоры: 0.8179
  мариновать: 0.6799
  здоровое: 0.5280
  начинка: 0.4470
  дрожжи: 0.3623

Размерность 50:
  помидоры: 0.7180
  вода: 0.3271
  месить: 0.3131
  салат: 0.2698
  суп: 0.2654

Размерность 100:
  помидоры: 0.7107
  готовить: 0.3567
  парить: 0.1933
  сахар: 0.1654
  десерт: 0.1581


Все модели более-менее справились со словом *помидор*. Для *кастрюли* наилучшие результаты показала модель с 10 размерностью, результаты для *сыра* нельзя назвать удачными